# XD2-Net (hierarchical) — 05: Explanations

Visualises the outputs of `scripts/run_experiment.py --model hierarchical` for any dataset:

- **Level 1, hierarchical (internal self-attention):**
  α — attention the case pays to each event (`[CASE]` row);
  β — attention each event pays to its fields (`[EVT]` row);
  joint α·β per (event, field), which sums to 1 per case.
- **Level 2:** thetas → exact logit contributions (unchanged from the baseline).

**Prerequisite:** run the experiment first, e.g.
```
python scripts/run_experiment.py --model baseline hierarchical \
    --datasets configs/datasets/bpic2012.yaml --buckets 1_10
```
Attention describes what went into the case representation, not a causal effect on the
prediction (see `docs/hierarchical_attention/APPROACH.md` §7).

In [ ]:
import os, sys, glob, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Colab: clone the repo into /content (code) and write outputs to Drive (they survive the runtime).
REPO_URL = 'https://github.com/bemali/xd2-net-pytorch.git'
BRANCH = 'feattransformer-self-attention-exp'       # switch to 'main' once merged
OUTPUT_ROOT = None                                   # None = configs' workspace_dir (./outputs)

if IN_COLAB:
    REPO_ROOT = '/content/xd2-net-pytorch'
    if not os.path.isdir(os.path.join(REPO_ROOT, '.git')):
        !git clone --branch {BRANCH} {REPO_URL} {REPO_ROOT}
    %pip install -e {REPO_ROOT} --quiet
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = '/content/drive/MyDrive/xd2net_outputs'
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
else:
    REPO_ROOT = os.path.abspath('..')

sys.path.insert(0, REPO_ROOT)   # importable without a kernel restart after pip install -e
os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT, '| commit:',
      subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout.strip())
print('OUTPUT_ROOT:', OUTPUT_ROOT)
try:
    import torch
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU runtime)')
except ImportError:
    pass

In [ ]:
# --- parameters --------------------------------------------------------------
WORKSPACE = OUTPUT_ROOT or './outputs'   # workspace_dir used by the runner
MODEL     = 'hierarchical'       # folder under WORKSPACE (the per-field view needs beta)
DATASET   = 'bpic2012'           # dataset name (folder under WORKSPACE/MODEL)
BUCKET    = '1_10'               # prefix bucket to inspect
CASE_ID   = None                 # None = first test case; or a specific case id (string)
TOP_N     = 20

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ds_dir  = os.path.join(WORKSPACE, MODEL, DATASET)
exp_dir = os.path.join(ds_dir, 'explanations')

cells = pd.read_csv(os.path.join(exp_dir, f'attention_cells_{BUCKET}.csv'), dtype={'CASEID': str})
case_fields = pd.read_csv(os.path.join(exp_dir, f'attention_case_fields_{BUCKET}.csv'), dtype={'CASEID': str})
preds = pd.read_csv(os.path.join(exp_dir, f'predictions_{BUCKET}.csv'), dtype={'CASEID': str})
level2 = pd.read_csv(os.path.join(exp_dir, f'explanations_{BUCKET}.csv'), dtype={'CASEID': str})
with open(os.path.join(ds_dir, 'dataset_summary.json')) as f:
    summary = json.load(f)

FIELDS = list(dict.fromkeys(cells['FIELD']))      # schema order
print(f"{DATASET} / {BUCKET}: {cells.CASEID.nunique()} test cases, fields = {FIELDS}")

## 1. Predictive performance across datasets and buckets

In [ ]:
results = pd.read_csv(os.path.join(WORKSPACE, 'results_all.csv'))
display(results)

fig, ax = plt.subplots(figsize=(8, 3.5))
for (model, name), g in results.groupby(['model', 'dataset']):
    g = g.assign(n=g['bucket'].str.split('_').str[1].astype(int)).sort_values('n')
    ax.plot(g['n'], g['roc_auc'], marker='o', linestyle='-' if model == MODEL else '--',
            label=f'{model} / {name}')
ax.set_xlabel('prefix length'); ax.set_ylabel('ROC AUC'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('XD2-Net — AUC per bucket')
plt.tight_layout(); plt.show()

# Side by side for this dataset (one AUC column per model)
display(results[results.dataset == DATASET].pivot(index='bucket', columns='model', values='roc_auc'))

## 2. One case: joint attention heatmap (events × fields)

Rows are events in chronological order, labelled with their activity. Each cell is α[t]·β[t,f].
Categorical cells are annotated with the label the event carried; numeric cells with the
model-space (normalised) value.

In [ ]:
case = CASE_ID if CASE_ID is not None else cells.CASEID.iloc[0]
c = cells[cells.CASEID == str(case)]
p = preds[preds.CASEID == str(case)].iloc[0]

joint = c.pivot(index='EVENT_INDEX', columns='FIELD', values='JOINT_ATTN')[FIELDS]
annot = c.assign(TXT=np.where(c.VALUE_LABEL.fillna('') != '', c.VALUE_LABEL.fillna(''),
                              c.VALUE_NUM.map(lambda v: f'{v:.2f}')))
annot = annot.pivot(index='EVENT_INDEX', columns='FIELD', values='TXT')[FIELDS]
ev = c.drop_duplicates('EVENT_INDEX').set_index('EVENT_INDEX')

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(14, 0.45 * len(joint) + 2),
                               gridspec_kw={'width_ratios': [4, 1]}, sharey=True)
sns.heatmap(joint, annot=annot, fmt='', cmap='Blues', ax=ax0, cbar_kws={'label': 'α·β'},
            annot_kws={'fontsize': 7})
ax0.set_yticklabels([f"{i}: {ev.loc[i, 'EVENT_ACTIVITY']}" for i in joint.index], rotation=0)
ax0.set_ylabel('event (chronological)'); ax0.set_xlabel('')
ax0.set_title(f"case {case} — p={p.PREDICTION_PROB:.2f}, truth={p.GROUND_TRUTH}")
ax1.barh(np.arange(len(ev)) + 0.5, ev['EVENT_ATTN'], color='tab:blue')
ax1.set_title('α (case → event)'); ax1.set_xlabel('attention')
plt.tight_layout(); plt.show()

# β per event (rows sum to 1)
beta = c.pivot(index='EVENT_INDEX', columns='FIELD', values='FEATURE_ATTN')[FIELDS]
display(beta.style.background_gradient(cmap='Greens', axis=1).format('{:.2f}'))

## 3. Global attention: per field and per (field, label)

In [ ]:
from xd2net.common.explainer import Explainer

g_field = Explainer.global_attention(cells, by='field')
g_label = Explainer.global_attention(cells, by='field_label')
g_label = g_label.assign(NAME=np.where(g_label.VALUE_LABEL.fillna('') != '',
                                        g_label.FIELD + '=' + g_label.VALUE_LABEL.fillna(''), g_label.FIELD))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=g_field, y='FIELD', x='ATTENTION', ax=axes[0], color='tab:blue')
axes[0].set_title('mean per-case attention by field')
top = g_label.head(TOP_N)
sns.barplot(data=top, y='NAME', x='ATTENTION', ax=axes[1], color='tab:orange')
axes[1].set_title(f'top {TOP_N} (field, label)')
plt.tight_layout(); plt.show()

# Attention by event position (chronological), averaged over cases
pos = cells.drop_duplicates(['CASEID', 'EVENT_INDEX']).groupby('EVENT_INDEX').EVENT_ATTN.mean()
pos.plot(kind='bar', figsize=(8, 3), title='mean α by event index (0 = first event)', color='tab:blue')
plt.tight_layout(); plt.show()

## 4. Diagnostics

* **Normalised entropy** of α and β (1 = uniform, 0 = all mass on one token). Values close to 1
  for every case mean the attention is uninformative.
* **Self-mass** of the `[EVT]` / `[CASE]` tokens. It is 0 by construction when
  `cls_self_attend: false` (the default). If enabled, a large self-mass means the summary relies
  on its learned prior rather than the inputs.

In [ ]:
def norm_entropy(w):
    w = np.asarray(w, dtype=float)
    w = w[w > 0]
    return 0.0 if len(w) <= 1 else float(-(w * np.log(w)).sum() / np.log(len(w)))

events = cells.drop_duplicates(['CASEID', 'EVENT_INDEX'])
h_alpha = events.groupby('CASEID').EVENT_ATTN.apply(norm_entropy)
h_beta = cells.groupby(['CASEID', 'EVENT_INDEX']).FEATURE_ATTN.apply(norm_entropy)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
axes[0].hist(h_alpha, bins=30, color='tab:blue'); axes[0].set_title('normalised entropy of α (per case)')
axes[1].hist(h_beta, bins=30, color='tab:green'); axes[1].set_title('normalised entropy of β (per event)')
axes[2].hist(events.EVT_SELF_ATTN, bins=30, alpha=0.7, label='[EVT]')
axes[2].hist(case_fields.drop_duplicates('CASEID').CASE_SELF_ATTN, bins=30, alpha=0.7, label='[CASE]')
axes[2].set_title('CLS self-mass'); axes[2].legend()
plt.tight_layout(); plt.show()
print(f"mean normalised entropy  α: {h_alpha.mean():.3f}   β: {h_beta.mean():.3f}")

## 5. Level-2: thetas (exact attribution of the prediction)

In [ ]:
g2 = (level2.groupby('FEATURE').FEATURE_IMPORTANCE.mean()
      .sort_values(ascending=False).head(TOP_N))
g2.iloc[::-1].plot(kind='barh', figsize=(8, 0.3 * len(g2) + 1), color='tab:purple',
                   title='mean theta × final weight (top features)')
plt.tight_layout(); plt.show()

case_l2 = level2[level2.CASEID == str(case)].sort_values('FEATURE_IMPORTANCE', ascending=False).head(10)
display(case_l2[['FEATURE', 'FEATURE_VALUE', 'FEATURE_IMPORTANCE']])